## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

## Setting up your keys - OPTIONAL!

We're now going to try asking a bunch of models some questions!

This is totally optional. If you have keys to Anthropic, Gemini or others, then you can add them in.

If you'd rather not spend the extra, then just watch me do it!

For OpenAI, visit https://openai.com/api/  
For Anthropic, visit https://console.anthropic.com/  
For Google, visit https://aistudio.google.com/   
For DeepSeek, visit https://platform.deepseek.com/  
For Groq, visit https://console.groq.com/  
For Grok, visit https://console.x.ai/  


You can also use OpenRouter as your one-stop-shop for many of these! OpenRouter is "the unified interface for LLMs":

For OpenRouter, visit https://openrouter.ai/  


With each of the above, you typically have to navigate to:
1. Their billing page to add the minimum top-up (except Gemini, Groq, Google, OpenRouter may have free tiers)
2. Their API key page to collect your API key

### Adding API keys to your .env file

When you get your API keys, you need to set them as environment variables by adding them to your `.env` file.

```
OPENAI_API_KEY=xxxx
ANTHROPIC_API_KEY=xxxx
GOOGLE_API_KEY=xxxx
DEEPSEEK_API_KEY=xxxx
GROQ_API_KEY=xxxx
GROK_API_KEY=xxxx
OPENROUTER_API_KEY=xxxx
```
Any time you change your .env file
Remember to Save it! And also rerun load_dotenv(override=True)


In [1]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [19]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if deepseek_api_key:
    print(f"DeepSeek API Key is set")
else:
    print("DeepSeek API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AQ
DeepSeek API Key exists and begins sk-
Groq API Key exists and begins gsk_
Grok API Key not set (and this is optional)
OpenRouter API Key exists and begins sk-


In [20]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

# For Gemini, DeepSeek and Groq, we can use the OpenAI python client
# Because Google and DeepSeek have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
deepseek_url = "https://api.deepseek.com"
groq_url = "https://api.groq.com/openai/v1"
grok_url = "https://api.x.ai/v1"
openrouter_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

In [4]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [5]:
response = openai.chat.completions.create(model="gpt-6-luna", reasoning_effort="none", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

Why did the LLM engineering student bring a ladder to the prompt?

They heard the model’s performance improved with a little *fine-tuning*—and wanted to reach the next level.

In [6]:
response = deepseek.chat.completions.create(model="deepseek-v4.1-flash", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

AuthenticationError: Error code: 401 - {'error': {'code': 'authentication_error', 'message': 'Invalid Anthropic API Key', 'type': 'invalid_request_error', 'param': None}}

## Training vs Inference time scaling

In [7]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [8]:
response = openai.chat.completions.create(model="gpt-6-luna", messages=easy_puzzle, reasoning_effort="none")
display(Markdown(response.choices[0].message.content))

\(\frac{2}{3}\)

In [9]:
response = openai.chat.completions.create(model="gpt-6-luna", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

2/3

In [10]:
response = openai.chat.completions.create(model="gpt-6.1-sol", messages=easy_puzzle, reasoning_effort="low")
display(Markdown(response.choices[0].message.content))

\(\frac{2}{3}\)

## Testing out the best models on the planet

In [11]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [12]:
response = openai.chat.completions.create(model="gpt-6-luna", messages=hard_puzzle, reasoning_effort="none")
display(Markdown(response.choices[0].message.content))

The worm travels through the two covers on the *inside* edges of the volumes, plus the pages.

With the books side by side in normal order, the first page of volume 1 is next to its front cover, while the last page of volume 2 is next to its back cover. Thus the route crosses the pages of both volumes and the two covers between them:

\[
2\text{ cm}+2\text{ cm}+2\text{ mm}+2\text{ mm}=4.4\text{ cm}.
\]

So it gnawed **4.4 cm**.

In [ ]:
response = deepseek.chat.completions.create(model="deepseek-v4.1-flash", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

In [13]:
response = openai.chat.completions.create(model="gpt-6.1-sol", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

**4 mm.**

With the books shelved normally, volume 1 on the left, its **first page** is next to volume 2. Volume 2’s **last page** is next to volume 1.

So the worm passes through only two covers—the front cover of volume 1 and the back cover of volume 2:

\[
2\text{ mm}+2\text{ mm}=\boxed{4\text{ mm}}.
\]

In [16]:
response = deepseek.chat.completions.create(model="deepseek-v4.1-flash", messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

The worm gnaws only through:

- the front cover of the first volume: \(2\) mm  
- the back cover of the second volume: \(2\) mm  

It does **not** gnaw through the pages, because the first page of volume 1 and the last page of volume 2 face inward toward each other when the volumes stand normally on the shelf.

\[
2\text{ mm} + 2\text{ mm} = \boxed{4\text{ mm} \approx 0.4\text{ cm}}
\]

**Answer: 4 mm.**

## A spicy challenge to test the competitive spirit

In [ ]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [ ]:
response = deepseek.chat.completions.create(model="deepseek-v4.1-flash", messages=dilemma)
display(Markdown(response.choices[0].message.content))


In [ ]:
response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = deepseek.chat.completions.create(model="deepseek-v4.1-flash", messages=dilemma)
display(Markdown(response.choices[0].message.content))

In [ ]:
response = grok.chat.completions.create(model="grok-4.3", messages=dilemma)
display(Markdown(response.choices[0].message.content))

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

In [ ]:
!ollama pull llama3.2

In [ ]:
# Only do this if you have a large machine - at least 16GB RAM

!ollama pull gpt-oss:20b

In [34]:
hamlet_question = [
    {"role": "user", "content": 
        "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"},
]
response = ollama.chat.completions.create(model="llama3.2:1b", messages=hamlet_question)
display(Markdown(response.choices[0].message.content))

When Laertes asks "Where is my father?" in the play Hamlet, according to his actual phrase delivered by Polonius in Act 1 Scene 2, and later repeated in Act 3 Scene 4, he says: "My father was laid ere either of you were born."

In [ ]:
response = ollama.chat.completions.create(model="gpt-oss:20b", messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

## Gemini and Anthropic Client Library

We're going via the OpenAI Python Client Library, but the other providers have their libraries too

In [18]:
from google import genai

client = genai.Client()

response = client.models.generate_content(
    model="gemini-3.1-flash-lite", contents="Describe the color Blue to someone who's never been able to see in 1 sentence"
)
print(response.text)

Blue is the sensation of a cool, steady breeze on your skin or the deep, calming stillness of silence after the noise of the day has faded.


In [ ]:
response = deepseek.chat.completions.create(
    model="deepseek-v4.1-flash",
    messages=[{"role": "user", "content": "Describe the color Blue to someone who's never been able to see in 1 sentence"}],
    max_tokens=100
)
print(response.choices[0].message.content)

## Routers and Abtraction Layers

Starting with the wonderful OpenRouter.ai - it can connect to all the models above!

Visit openrouter.ai and browse the models.

Here's one we haven't seen yet: GLM 4.5 from Chinese startup z.ai

In [22]:
response = openrouter.chat.completions.create(model="nvidia/nemotron-3-ultra-550b-a55b:free", messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))

A student asks their professor, **"What is the single most important hyperparameter to tune when trying to become an LLM Expert?"**

The professor thinks for a moment, adjusts their GPU-cluster-shaped glasses, and replies:

> **"Your **Learning Rate**.**
>
> **If it’s too high, you diverge into chaos (burnout).**
> **If it’s too low, you converge *way* too slowly (tutorial hell).**
> **And whatever you do—don’t forget to clip your **Gradients** (expectations), or you’ll explode your **Memory** (budget) before you even reach the first **Checkpoint** (job offer)."**

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-6-luna", reasoning_effort="none")
response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

Why did the LLM engineering student bring a ladder to class?

They heard the course was about taking prompts to the next level.

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [25]:
from litellm import completion
response = completion(model="openai/gpt-4.1", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Sure! Here’s a joke just for budding LLM Engineers:

Why did the student bring a ladder to the LLM lab?

Because they heard it’s the only way to reach new “model heights”!

In [26]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 24
Output tokens: 41
Total tokens: 65
Total cost: 0.0376 cents


## Now - let's use LiteLLM to illustrate a Pro-feature: prompt caching

In [27]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [28]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [40]:
question

[{'role': 'user',
  'content': "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?\n\nFor context, here is the entire text of Hamlet:\n\nSpeak, man.\n  Laer. Where is my father?\n  King. Dead.\n  Queen. But not by him!\n  King. Let him deman"}]

In [29]:
response = completion(model="gemini/gemini-3.1-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

When Laertes demands to know where his father (Polonius) is, Claudius replies:

**"Dead."**

Gertrude quickly adds, **"But not by him."** (referring to Claudius).

This exchange occurs in Act 4, Scene 5, when Laertes storms the castle and confronts the King, believing Claudius killed his father.

In [30]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 19
Output tokens: 78
Total tokens: 97
Total cost: 0.0122 cents


In [35]:
hamlets_part=hamlet[loc:loc+100]

In [36]:
question

[{'role': 'user',
  'content': "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [ ]:
question[0]["content"] += "\n\nFor context, here is the some text of Hamlet:\n\n"+hamlets_part

In [38]:
question

[{'role': 'user',
  'content': "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?\n\nFor context, here is the entire text of Hamlet:\n\nSpeak, man.\n  Laer. Where is my father?\n  King. Dead.\n  Queen. But not by him!\n  King. Let him deman"}]

In [ ]:

response = ollama.chat.completions.create(model="llama3.2:1b", messages=question)
display(Markdown(response.choices[0].message.content))

In [50]:
response = completion(model="openai/gpt-6-luna", messages=question)
display(Markdown(response.choices[0].message.content))

The King replies, “Dead.”

In [51]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 68
Output tokens: 45
Cached tokens: 0
Total cost: 0.0029 cents


In [52]:
response = completion(model="openai/gpt-6-luna", messages=question)
display(Markdown(response.choices[0].message.content))

The King replies, “Dead.”

In [53]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 68
Output tokens: 34
Cached tokens: 0
Total cost: 0.0024 cents


## Prompt Caching with OpenAI

For OpenAI:

https://platform.openai.com/docs/guides/prompt-caching

> Cache hits are only possible for exact prefix matches within a prompt. To realize caching benefits, place static content like instructions and examples at the beginning of your prompt, and put variable content, such as user-specific information, at the end. This also applies to images and tools, which must be identical between requests.


Cached input is 4X cheaper

https://openai.com/api/pricing/

## Prompt Caching with Anthropic

https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

You have to tell Claude what you are caching

You pay 25% MORE to "prime" the cache

Then you pay 10X less to reuse from the cache with inputs.

https://www.anthropic.com/pricing#api

## Gemini supports both 'implicit' and 'explicit' prompt caching

https://ai.google.dev/gemini-api/docs/caching?lang=python

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [ ]:
# Let's make a conversation between GPT-6 Luna and DeepSeek V4.1 Flash
# We're using cheap versions of models so the costs will be minimal

gpt_model = "gpt-6-luna"
deepseek_model = "deepseek-v4.1-flash"

gpt_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

deepseek_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

gpt_messages = ["Hi there"]
deepseek_messages = ["Hi"]

In [ ]:
def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    for gpt, deepseek in zip(gpt_messages, deepseek_messages):
        messages.append({"role": "assistant", "content": gpt})
        messages.append({"role": "user", "content": deepseek})
    response = openai.chat.completions.create(model=gpt_model, reasoning_effort="none", messages=messages)
    return response.choices[0].message.content

In [ ]:
call_gpt()

In [ ]:
def call_deepseek():
    messages = [{"role": "system", "content": deepseek_system}]
    for gpt, deepseek_message in zip(gpt_messages, deepseek_messages):
        messages.append({"role": "user", "content": gpt})
        messages.append({"role": "assistant", "content": deepseek_message})
    messages.append({"role": "user", "content": gpt_messages[-1]})
    response = deepseek.chat.completions.create(model=deepseek_model, messages=messages)
    return response.choices[0].message.content

In [ ]:
call_deepseek()

In [ ]:
call_gpt()

In [ ]:
gpt_messages = ["Hi there"]
deepseek_messages = ["Hi"]

display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### DeepSeek:\n{deepseek_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    deepseek_next = call_deepseek()
    display(Markdown(f"### DeepSeek:\n{deepseek_next}\n"))
    deepseek_messages.append(deepseek_next)

## Before you continue

Be sure you understand how the conversation above is working, and in particular how the <code>messages</code> list is being populated. Add print statements as needed. Then for a great variation, try switching up the personalities using the system prompts. Perhaps one can be pessimistic, and one optimistic?

# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

## Business relevance

This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.